# Flight Delay and Overbooking Prediction
**Course:** Advanced AI -- 2025/2026  
**Author:** Moad Khaouili

Two binary classification models:
1. **Flight Delay Prediction** -- will a flight depart more than 15 minutes late?
2. **Overbooking Prediction** -- is a flight overbooked?

---

## Section 1 -- Data Preparation

> **All data files have already been prepared and saved to `data/`. This section explains what was done -- no need to re-run.**

---

Two datasets were combined to build the training data:

- **Flight delay data** (`full_data_flightdelay.csv`): ~6.5M US domestic flights from 2019. The last 10,000 rows were extracted as `sample_10k_flightdelay.csv`. Contains the delay target (`DEP_DEL15`) and flight/weather features.
- **Synthetic booking data** (`flight_bookings.csv`): ~366k simulated flight records generated from public SFO passenger statistics, containing seat counts, passenger counts, load factors, and an overbooking label.

**These two datasets were merged on: airline name + month.**

Before joining, carrier names were normalized -- lowercased and stripped of punctuation and legal suffixes such as Inc., Co., Ltd. -- so names like `Delta Air Lines Inc.` and `Delta Air Lines` would match correctly. The bookings data was then aggregated per (airline, month), computing average load factor, average passengers, and overbooking rate, and left-joined onto the delay sample.

81.4% of rows found a match. The remaining 18.6% (carriers not in the SFO dataset, e.g. United, Spirit) have NaN for booking columns and will be imputed before training.

**Output:** `merged_flights.csv` -- 10,000 rows x 30 columns.

## Section 2 -- Load Dataset

Load the pre-built merged dataset and verify its contents before feature engineering.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/merged_flights.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
print("=== Dataset Summary ===")
print(f"Total rows: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print("\n--- Delay Target (DEP_DEL15) ---")
print(df["DEP_DEL15"].value_counts(normalize=True).round(3))
print("\n--- Booking Feature Coverage ---")
booking_cols = ["avg_load_factor", "avg_passengers", "overbooked_rate", "avg_seats_booked"]
print(df[booking_cols].notna().mean().round(3))
print("\n--- Null counts (columns with nulls only) ---")
nulls = df.isnull().sum()
print(nulls[nulls > 0])

## Section 3 -- ASRS LLM Risk Score Feature

This section adds an optional feature derived from the **Aviation Safety Reporting System (ASRS)** dataset. ASRS collects voluntary incident reports submitted by pilots, controllers, and crew -- each containing a free-text narrative describing a safety-related event.

### Concept

Each incident narrative is passed through an LLM (zero-shot classifier) which returns a **risk score between 0 and 1** reflecting how likely the described event is to contribute to a flight delay. Scores are aggregated by airport and month, then merged into the main dataset. Flights with no matching ASRS report are assigned a score of **0** (no incident reported).

### Why this helps

Standard features like weather or congestion capture systemic delay causes. ASRS narratives capture rare but high-impact events -- equipment failures, near-misses, ATC issues -- that would otherwise be invisible to the model. The LLM converts unstructured text into a single interpretable numeric feature.

### Merge key

The ASRS score is joined onto the delay dataset using:
- `DEPARTING_AIRPORT` -- departure airport name
- `MONTH` -- month of the flight (1-12)

When multiple reports exist for the same airport+month, their scores are averaged.

### Pipeline

```
ASRS narratives (free text)
        |
        v
LLM zero-shot classifier
        |
        v
risk_score per report (0 to 1)
        |
        v
Aggregate: mean score per (airport, month)
        |
        v
Left join onto merged_flights.csv
        |
        v
Fill missing with 0  (no incident reported)
```

In [ ]:
from transformers import pipeline

# Load zero-shot classification model as the LLM risk scorer
scorer = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

CANDIDATE_LABELS = [
    "likely to cause a flight delay",
    "unlikely to cause a flight delay"
]

def score_narrative(text):
    """Return delay risk score (0-1) for a single ASRS narrative."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return 0.0
    result = scorer(text[:512], candidate_labels=CANDIDATE_LABELS)
    return round(result["scores"][0], 4)

print("LLM scorer loaded.")

### Load and Score ASRS Reports

Expected columns in the ASRS CSV:

| Column | Description |
|---|---|
| `narrative` | Free-text incident report |
| `airport` | Airport name matching `DEPARTING_AIRPORT` |
| `month` | Month of the incident (1-12) |

A small demo sample is used below to illustrate the pipeline.

In [ ]:
# To use real ASRS data, replace this with:
# df_asrs = pd.read_csv("../data/asrs_reports.csv")

# --- Demo sample ---
df_asrs = pd.DataFrame({
    "narrative": [
        "Aircraft experienced hydraulic failure on taxiway, causing significant ground delay.",
        "Routine flight, no issues reported.",
        "ATC miscommunication led to runway incursion, departure held for 40 minutes.",
        "Bird strike on approach, aircraft returned to gate for inspection.",
        "Normal operations, weather clear.",
    ],
    "airport": [
        "McCarran International",
        "Orlando International",
        "McCarran International",
        "Boise Air Terminal",
        "Orlando International",
    ],
    "month": [1, 1, 3, 2, 2]
})

df_asrs["risk_score"] = df_asrs["narrative"].apply(score_narrative)
df_asrs[["airport", "month", "risk_score", "narrative"]]

In [ ]:
# Aggregate: mean risk score per (airport, month)
asrs_agg = df_asrs.groupby(["airport", "month"])["risk_score"].mean().reset_index()
asrs_agg.columns = ["DEPARTING_AIRPORT", "MONTH", "asrs_risk_score"]
asrs_agg["asrs_risk_score"] = asrs_agg["asrs_risk_score"].round(4)

# Left join onto main dataset
df = df.merge(asrs_agg, on=["DEPARTING_AIRPORT", "MONTH"], how="left")

# Flights with no ASRS report get score 0
df["asrs_risk_score"] = df["asrs_risk_score"].fillna(0.0)

print(f"asrs_risk_score added.")
print(f"Non-zero entries: {(df['asrs_risk_score'] > 0).sum()} / {len(df)}")
print(df["asrs_risk_score"].describe().round(4))